# Основная часть

## Установка файлов

Для работы программы необходимо загрузить в Колаб следующие файлы:
- `scaffolds.fasta` -- fasta-файл со скаффолдами.
- `gms2.lst` -- координаты всех предсказанных генов;
- `proteins.fasta` -- аминокислотные последовательности всех предсказанных генов;
- `scaffolds.hits_from_MIL_1.txt` -- информация о схожести белков из нашей бактерии с белками из бактерии MIL-1;
- `scaffolds.hits_from_SwissProt.txt` -- информация о схожести белков из нашей бактерии с белками из БД SwissProt.

## Установка E-utilities (для скачивания последовательностей из NCBI)

In [1]:
!sh -c "$(curl -fsSL ftp://ftp.ncbi.nlm.nih.gov/entrez/entrezdirect/install-edirect.sh)"


Entrez Direct has been successfully downloaded and installed.

In order to complete the configuration process, please execute the following:

  echo "export PATH=/root/edirect:\${PATH}" >> ${HOME}/.bashrc

or manually edit the PATH variable assignment in your .bashrc file.

Would you like to do that automatically now? [y/N]
^C


## Скачиваем данные по близкородственной бактерии T.oleivorans

In [2]:
!$HOME/edirect/efetch -db nuccore -id HF680312 -format gb  >  T_oleivorans_MIL_1.gbk

## Аннотация генома

Установка библиотеки BioPython

In [3]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.3 MB/s eta 0:00:00


Импорты библиотек

In [4]:
from Bio import SeqIO
from Bio.SeqFeature import SeqFeature, FeatureLocation
from datetime import datetime as dt
import pandas as pd

Аннотация скаффолдов

In [5]:
scaffolds = dict()

for record in SeqIO.parse("scaffolds.fasta", "fasta"):
    record.annotations['molecule_type'] = 'DNA'
    record.annotations['date'] = dt.now().strftime("%d-%b-%Y").upper()
    record.annotations['data_file_division'] = 'BCT'
    scaffolds[record.id] = record

Аннотация генов

In [6]:
genes = dict()

for gene in SeqIO.parse("proteins.fasta", "fasta"):
    desc = gene.description.split(' ')
    scaffold = desc[1]
    start, end = int(desc[2]), int(desc[3])
    strand = 1 if desc[4] == '+' else -1
    feat = SeqFeature(FeatureLocation(start, end, strand=strand), type="CDS")
    feat.qualifiers['locus_tag'] = [desc[0]]
    feat.qualifiers['translation'] = [gene.seq]
    scaffolds[scaffold].features.append(feat)
    genes[desc[0]] = feat

### Добавление функции белков из MIL-1

In [7]:
genbank_record = SeqIO.read("T_oleivorans_MIL_1.gbk", "genbank")
mil_genes = {
    feature.qualifiers['protein_id'][0]: feature.qualifiers['product'][0]
    for feature in genbank_record.features
    if 'protein_id' in feature.qualifiers and 'product' in feature.qualifiers
}

In [8]:
columns = [
    'qseqid',
    'sseqid',
    'pident',
    'length',
    'mismatch',
    'gapopen',
    'qstart',
    'qend',
    'sstart',
    'send',
    'evalue',
    'bitscore'
]

mil_hits = pd.read_csv(
    'scaffolds.hits_from_MIL_1.txt',
    sep='\t', header=None, names=columns
)

mil_hits.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,1,lcl|HF680312.1_prot_CCU72855.1_2406,99.545,220,1,0,1,220,1,220,1.430000e-167,457.0
1,2,lcl|HF680312.1_prot_CCU72856.1_2407,100.000,234,0,0,1,234,1,234,3.460000e-180,490.0
2,3,lcl|HF680312.1_prot_CCU72857.1_2408,99.375,320,2,0,1,320,1,320,0.000000e+00,643.0
3,4,lcl|HF680312.1_prot_CCU72858.1_2409,100.000,273,0,0,10,282,1,273,0.000000e+00,562.0
4,4,lcl|HF680312.1_prot_CCU72327.1_1878,28.322,286,185,4,1,280,1,272,1.260000e-30,112.0


Находим попадания

In [9]:
hits = mil_hits[
    mil_hits['sseqid']
    .str
    .contains("CCU")
].sort_values('bitscore', ascending=False).drop_duplicates('qseqid')

hits.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
3649,1273,lcl|HF680312.1_prot_CCU73896.1_3447,98.979,2253,23,0,1,2253,1,2253,0.0,4296.0
2220,811,lcl|HF680312.1_prot_CCU70724.1_275,97.870,2113,43,1,1,2113,1,2111,0.0,4158.0
7335,2611,lcl|HF680312.1_prot_CCU71900.1_1451,99.878,1640,2,0,1,1640,1,1640,0.0,3406.0
3875,1349,lcl|HF680312.1_prot_CCU73861.1_3412,99.214,1654,13,0,1,1654,1,1654,0.0,3372.0
6557,2334,lcl|HF680312.1_prot_CCU71621.1_1172,99.690,1611,5,0,1,1611,1,1611,0.0,3338.0


In [10]:
for i, hit in hits.iterrows():
    gene = genes[str(hit['qseqid'])]
    _match = hit['sseqid'].split('_')[2]
    gene.qualifiers['product'] = [mil_genes[_match]]

### Добавление функции белков из SwissProt

Скачиваем базу UniProt

In [11]:
!wget -nc https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.dat.gz

--2024-10-26 13:44:25--  https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.dat.gz
Resolving ftp.uniprot.org (ftp.uniprot.org)... 128.175.240.195
Connecting to ftp.uniprot.org (ftp.uniprot.org)|128.175.240.195|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 671614860 (641M) [application/x-gzip]
Saving to: ‘uniprot_sprot.dat.gz’

uniprot_sprot.dat.g 100%[===================>] 640.50M  42.8MB/s    in 9.0s    

2024-10-26 13:44:34 (70.9 MB/s) - ‘uniprot_sprot.dat.gz’ saved [671614860/671614860]



In [12]:
!gzip -d uniprot_sprot.dat.gz

Отбираем по шаблону и сохраняем строки

In [13]:
!grep '^ID\|^DE   RecName: Full=' uniprot_sprot.dat > SwissProt_names.txt

Парсинг строк в файл формата GenBank

In [14]:
previd = None
swissgenes = dict()

for line in open('SwissProt_names.txt'):
    if line.startswith('ID'):
        previd = line.split()[1]
    if line.startswith('DE'):
        swissgenes[previd] = line.split('=')[1][:-2]

In [15]:
swisshits_raw = pd.read_csv(
    'scaffolds.hits_from_SwissProt.txt', sep='\t',
    header=None, names=columns
)

swisshits = swisshits_raw \
                .sort_values('bitscore', ascending=False) \
                .drop_duplicates('qseqid')

for i, hit in swisshits.iterrows():
    gene = genes[str(hit['qseqid'])]
    _match = hit['sseqid'].split('|')[-1]
    gene.qualifiers['product'] = [swissgenes[_match]]

In [16]:
SeqIO.write(scaffolds.values(), "GENOME.gbk", "genbank")

80

# Бонусная часть

## Установка програм

In [17]:
!apt-get update

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,071 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
G

In [18]:
!apt-get install ncbi-blast+

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  liblmdb0 ncbi-data
The following NEW packages will be installed:
  liblmdb0 ncbi-blast+ ncbi-data
0 upgraded, 3 newly installed, 0 to remove and 50 not upgraded.
Need to get 15.9 MB of archives.
After this operation, 71.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 liblmdb0 amd64 0.9.24-1build2 [47.6 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-data all 6.1.20170106+dfsg1-9 [3,519 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-blast+ amd64 2.12.0+ds-3build1 [12.3 MB]
Fetched 15.9 MB in 0s (36.1 MB/s)
Selecting previously unselected package liblmdb0:amd64.
(Reading database ... 123622 files and directories currently installed.)
Preparing to unpack .../liblmdb0_0.9.24-1build2_amd64.deb ...
Unpacking liblmdb0:amd64 (0.9.24-1build2) ...
Selec

## Предсказание рибосомальной рнк

Проходимся по геному `T_oleivorans_MIL_1.gbk`, и записываем координаты начала, конца и знака стренда в рРНК

In [20]:
coords_start = []
coords_end = []
coords_strand = []
data = SeqIO.read("T_oleivorans_MIL_1.gbk", "genbank")

for f in data.features:
    if f.type == 'rRNA':
        coords_start.append(int(f.location.start))
        coords_end.append(int(f.location.end))
        coords_strand.append(f.location.strand)

In [21]:
with open('result.fasta', 'w') as f:
    for i in range(len(coords_strand)):
        f.write(f'> rRNA {coords_start[i]}...{coords_end[i]}\n')
        f.write(str(data.seq[coords_start[i]:coords_end[i]]) + '\n')

In [22]:
!blastn -query result.fasta -subject scaffolds.fasta > info.gbk

In [31]:
f = open("info.gbk", 'r')
for_look = False
al = []
percent = []
cur = []
choice = ("Query=", ">", "Identities")

for line in f:
    line = line.split()
    if not line or line[0] not in choice:
        continue

    if line[0] == choice[1]:
        if al:
            percent.append(cur)
        cur = []
        al.append(line[1])
        continue

    if line[0] == choice[2]:
        cur.append(line[3][1:-2:])
        continue

    if for_look == False:
        for_look = True
        remem = line[2]
    percent.append(cur)
    if al != []:
        print(f'Resemblance rRna {remem}')
        for i in range(len(al)):
            print(f"For seq = {al[i]} \t {percent[i][0]} resemblance")
        for_look == False
        print()

    al = []
    percent = []

f.close()

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 99% resemblance
For seq = scaffold70_cov665 	 100% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 99% resemblance
For seq = scaffold65_cov675 	 100% resemblance
For seq = scaffold63_cov665 	 100% resemblance
For seq = scaffold66_cov704 	 100% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 100% resemblance
For seq = scaffold65_cov675 	 100% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 98% resemblance
For seq = scaffold65_cov675 	 99% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 99% resemblance
For seq = scaffold65_cov675 	 100% resemblance
For seq = scaffold63_cov665 	 100% resemblance
For seq = scaffold66_cov704 	 100% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov273 	 99% resemblance
For seq = scaffold70_cov665 	 99% resemblance

Resemblance rRna 341494...343033
For seq = scaffold1_cov2